# Actualizar un usuario

## Qué hace

Cambia el nombre del usuario de demo a Jimena Demo Actualizada.

## Qué pasa en la base de datos

Actualiza su nombre y fecha de cambio; conserva el correo y el ID.

## Qué acción usa

`updateUser(user.id, { name: "Jimena Demo Actualizada" })`.

## Anotaciones

Solo modifica el usuario con el correo de demo. Si no existe, muestra User not found.

Selecciona la celda de abajo y pulsa **▶** o **Shift+Enter**. No necesitas
escribir código. El resultado aparece debajo en una tabla con encabezados cian.
Los datos de demostración ya están preparados en el documento.


In [ ]:
import json
import subprocess
from pathlib import Path

location = Path.cwd().resolve()
app = None
for directory in (location, *location.parents):
    candidate = directory / "staff/jimena-duran/first-app"
    source = candidate / "src"
    markers = ("logic/create-user.js", "queries/users.js", "db/connection.js")
    if (candidate / "package.json").is_file() and all((source / marker).is_file() for marker in markers):
        app = source
        break
    if (directory / ".git").exists():
        break
if app is None:
    raise SystemExit("No encuentro staff/jimena-duran/first-app/src. Abre Jupyter desde first-app, src o una subcarpeta de esta app.")

code = r"""
import { updateUser } from "./logic/update-user.js";
import { findUserByEmail } from "./logic/find-user-by-email.js";

try {
  const user = findUserByEmail("demo.jimena.jupyter@example.com");
  const result = updateUser(user.id, { name: "Jimena Demo Actualizada" });
  console.log(JSON.stringify(result === undefined ? null : result));
} catch (error) {
  console.error(error.message);
  process.exitCode = 1;
}
"""

try:
    completed = subprocess.run(
        ["node", "--input-type=module", "--eval", code],
        cwd=app,
        capture_output=True,
        encoding="utf-8",
    )
except OSError:
    raise SystemExit("No encuentro Node.js. Revisa la preparación de la app.")

if completed.returncode != 0:
    print(completed.stderr.strip())
else:
    result = json.loads(completed.stdout)
    headers = ['id', 'name', 'email', 'createdAt', 'updatedAt']
    rows = [[result[field] for field in headers]]
    # Tabla de texto con encabezados cian, sin librerías adicionales.
    rows = [[str(value).replace("\n", " ").replace("\r", " ").replace("\x1b", "") for value in row] for row in rows]
    widths = [max(len(header), *(len(row[index]) for row in rows)) if rows else len(header) for index, header in enumerate(headers)]
    print("┌" + "┬".join("─" * (width + 2) for width in widths) + "┐")
    print("│ " + " │ ".join("\x1b[1;36m" + header.ljust(width) + "\x1b[0m" for header, width in zip(headers, widths)) + " │")
    print("├" + "┼".join("─" * (width + 2) for width in widths) + "┤")
    for row in rows:
        print("│ " + " │ ".join(value.ljust(width) for value, width in zip(row, widths)) + " │")
    print("└" + "┴".join("─" * (width + 2) for width in widths) + "┘")
    if not rows:
        print("No hay filas.")
